In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text('incremental_flag','o')

In [0]:
incremental_flag = dbutils.widgets.get('incremental_flag')
print(incremental_flag)

In [0]:
df_src =spark.sql( 
    """
        SELECT Dealer_ID FROM PARQUET.`abfss://silver@carsalesdata.dfs.core.windows.net/carsales`
    """
)
df_src.display()

In [0]:
if not spark.catalog.tableExists('cars_catalog.gold.dim_dealer'):
    df_sink = spark.sql("""
        SELECT 1 AS dim_dealer_key, Dealer_ID FROM PARQUET.`abfss://silver@carsalesdata.dfs.core.windows.net/carsales`
        WHERE 1=0
    """
    )
else:
    df_sink = spark.sql("""
        SELECT dim_dealer_key, Dealer_ID FROM cars_catalog.gold.dim_dealer
    """)


In [0]:
df_sink.display()

In [0]:
df_filter = df_src.join(df_sink,df_src.Dealer_ID == df_sink.Dealer_ID,'left').select(df_src.Dealer_ID,df_sink.dim_dealer_key)
df_filter = df_filter.dropDuplicates(['dim_dealer_key'])
df_filter.display()


In [0]:
df_filter_old = df_filter.filter(df_filter.dim_dealer_key.isNotNull())
df_filter_old.display()

df_filter_new = df_filter.filter(df_filter.dim_dealer_key.isNull()).select('Dealer_ID')
df_filter_new.display()


In [0]:
if incremental_flag == 0:
    max_value =0
else:
    if spark.catalog.tableExists('cars_catalog.gold.dim_dealer'):
        max_value = spark.sql("""
                                SELECT MAX(dim_dealer_key) FROM cars_catalog.gold.dim_dealer
                              """).first()[0]
    else:
        max_value = 0
print(max_value)

In [0]:
from pyspark.sql import Window

window = Window.orderBy(lit(1))

df_filter_new = df_filter_new.withColumn('dim_dealer_key',row_number().over(window)+max_value)

df_filter_new.display()

In [0]:
df_final = df_filter_new.union(df_filter_old)
df_final.display()

In [0]:
from delta.tables import DeltaTable

if spark.catalog.tableExists('cars_catalog.gold.dim_dealer'):
    deltaTable =DeltaTable.forPath(spark,"abfss://gold@carsalesdata.dfs.core.windows.net/dim_dealer")

    deltaTable.alias('t').merge(df_final.alias('s'),"t.dim_dealer_key = s.dim_dealer_key")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

else:
    df_final.write.format('delta')\
        .mode('overwrite')\
        .option("path","abfss://gold@carsalesdata.dfs.core.windows.net/dim_dealer")\
        .saveAsTable('cars_catalog.gold.dim_dealer')


In [0]:
%sql
select * from cars_catalog.gold.dim_dealer;